*Adapted from code shared by David Gorkin (shared 9/29/25)*

**Use GENCODE v47 annotations to generate gene-level metadata (for tau index generation)**

# Settings

In [1]:
library(dplyr)
library(rtracklayer)
library(here)
here::i_am("figures/Figure2/tau/format_rowdata_human_gencode.ipynb")
sessionInfo()


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loading required package: GenomicRanges

Loading required package: stats4

Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:dplyr’:

    combine, intersect, setdiff, union


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, intersect, is.unsorted, lapply, Map, mapply,
    match, mget, order, paste, pmax, pmax.int, pmin, pmin.int,
    Position, rank, rbind, Reduce, rownames, sapply, setdiff, sort,
    table, tapply, union, unique, unsplit, which.max, which.min


Loading

R version 4.2.2 Patched (2022-11-10 r83330)
Platform: x86_64-pc-linux-gnu (64-bit)
Running under: Ubuntu 23.04

Matrix products: default
BLAS:   /usr/lib/x86_64-linux-gnu/openblas-pthread/libblas.so.3
LAPACK: /usr/lib/x86_64-linux-gnu/openblas-pthread/libopenblasp-r0.3.21.so

locale:
 [1] LC_CTYPE=C.UTF-8       LC_NUMERIC=C           LC_TIME=C.UTF-8       
 [4] LC_COLLATE=C.UTF-8     LC_MONETARY=C.UTF-8    LC_MESSAGES=C.UTF-8   
 [7] LC_PAPER=C.UTF-8       LC_NAME=C              LC_ADDRESS=C          
[10] LC_TELEPHONE=C         LC_MEASUREMENT=C.UTF-8 LC_IDENTIFICATION=C   

attached base packages:
[1] stats4    stats     graphics  grDevices utils     datasets  methods  
[8] base     

other attached packages:
[1] here_1.0.2           rtracklayer_1.58.0   GenomicRanges_1.50.2
[4] GenomeInfoDb_1.34.9  IRanges_2.32.0       S4Vectors_0.36.2    
[7] BiocGenerics_0.44.0  dplyr_1.1.4         

loaded via a namespace (and not attached):
 [1] SummarizedExperiment_1.28.0 pbdZMQ_0.3-13          

In [2]:
# Input files:

# - GENCODE reference file
gencode_file <- here(
  "inputs", "GENCODE",
  "gencode.v47.annotation.gtf.gz"
)

# Output file
outfile <- here(
  "inputs", "GENCODE",
  "v47_rowdata_human.rds"
)

# Load and format GENCODE reference

In [3]:
gencode <- import(gencode_file)

# subset gtf entries for genes
gencode <- gencode[which(gencode$type=="gene"),]

# create rowdata data frame
rowdata <- as.data.frame(gencode@elementMetadata)

# set rownames to gene ids
rownames(rowdata) <- rowdata$gene_id

# create column with "base" gene id (without the decimal gene version number)
rowdata$gene_id_base <- rownames(rowdata) %>%
    lapply(FUN=strsplit, split="\\.") %>%
    unlist() %>%
    matrix(ncol=2, byrow=T)  %>%
    subset.matrix(select = 1) %>%
    as.character()

#  eliminate columns with all NAs
rowdata <- rowdata[,colSums(is.na(rowdata))<nrow(rowdata)]

# add genome coordinates
rowdata$chr <- as.character(gencode@seqnames)
rowdata$start <- as.numeric(gencode@ranges@start)
rowdata$end <- rowdata$start + as.numeric(gencode@ranges@width)
#rm(gencode)

# view summary
print(paste("## Genes loaded: ", nrow(rowdata)))

[1] "## Genes loaded:  78724"


In [4]:
# Filter out pseudogenes
temp.filt <- grepl("pseudogene", rowdata$gene_type, ignore.case=TRUE)
print(paste("## Filter pseudogenes: ", length(which(temp.filt))))
rowdata <- rowdata[which(!temp.filt),]
print(paste("## Genes remaining: ", nrow(rowdata)))

[1] "## Filter pseudogenes:  15200"
[1] "## Genes remaining:  63524"


In [5]:
# Filter entries without valid hgnc_id, gene_id (to match rna-seq), and gene_names
temp.filt <- is.na(rowdata$hgnc_id) | is.na(rowdata$gene_id) | is.na(rowdata$gene_name)
print(paste("## Filter genes without valid identifiers: ", length(which(temp.filt))))
rowdata <- rowdata[which(!temp.filt),]
print(paste("## Genes remaining: ", nrow(rowdata)))

[1] "## Filter genes without valid identifiers:  32740"
[1] "## Genes remaining:  30784"


In [6]:
# confirm all gene_ids are unique.
!any(duplicated(rowdata$gene_id))

[1] TRUE

In [7]:
# confirm all gene_ids are unique. If not, will remove duplicate entries
!any(duplicated(rowdata$hgnc_id))

temp.filt <- rowdata$hgnc_id %in% rowdata$hgnc_id[which(duplicated(rowdata$hgnc_id))]
print(paste("## Filter entries with duplicate hgnc_id: ", length(which(temp.filt))))
rowdata <- rowdata[which(!temp.filt),]
print(paste("## Genes remaining: ", nrow(rowdata)))
!any(duplicated(rowdata$hgnc_id))

[1] FALSE

[1] "## Filter entries with duplicate hgnc_id:  88"
[1] "## Genes remaining:  30696"


[1] TRUE

In [8]:
# confirm all gene_names are unique.
!any(duplicated(rowdata$gene_name))

[1] TRUE

In [9]:
# confirm all gene_ids are unique.
!any(duplicated(rowdata$hgnc_id))

[1] TRUE

In [10]:
# confirm all gene_ids are unique.
!any(duplicated(rowdata$gene_symbol))

[1] TRUE

In [11]:
print(paste("## Final genes: ", nrow(rowdata)))

[1] "## Final genes:  30696"


In [12]:
#write out metadata
saveRDS(rowdata, file=outfile)